# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 8.3 MB/s eta 0:00:00
dependencies ok


In [4]:
import re
import gc
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference,helper,numpy_helper,TensorProto

In [5]:
TASK_ID = 'task096'
REVISION = "compact-v4-after-repeated-zero-score"

N = 30
K = 10


TASK_CANDIDATES = [
    Path(COMPETITION) / f"{TASK_ID}.json",
    Path.cwd() / f"{TASK_ID}.json",
    Path("/mnt/data") / f"{TASK_ID}.json",
]
TASK_PATH = next((p for p in TASK_CANDIDATES if p.exists()), None)
if TASK_PATH is None:
    raise FileNotFoundError(TASK_CANDIDATES)
with TASK_PATH.open() as f:
    task = json.load(f)

ADVERSARIAL_CASES = json.loads('[{"input":[[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,0,9,9,9,4,4,4,9,9,2,2,9,2,2,9,9,9,9,9,9,9],[9,9,9,9,9,4,9,4,9,9,2,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,4,4,4,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,2,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,2,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,7,7,7,9,7,7,7,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,1,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,1,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,1,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,1,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,1,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,1,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,1,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,1,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9]],"output":[[1,1,1,1,9,1,1,1,1],[1,7,7,7,9,7,7,7,1],[1,7,2,2,9,2,2,7,1],[1,7,2,4,4,4,2,7,1],[9,9,9,4,0,4,9,9,9],[1,7,2,4,4,4,2,7,1],[1,7,2,2,9,2,2,7,1],[1,7,7,7,9,7,7,7,1],[1,1,1,1,9,1,1,1,1]]},{"input":[[6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6],[6,8,8,8,6,6,3,3,6,3,3,6,0,6,6,6,6,6,6,6,6,6],[6,8,6,8,6,6,3,6,6,6,3,6,0,6,6,6,6,6,6,6,6,6],[6,8,8,8,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,3,6,6,6,3,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,3,3,6,3,3,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,6,6,0,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,6,6,0,6,6,6,6,6,6,6,6,6],[6,9,9,9,6,6,6,9,9,9,6,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,4,4,4,4,4,6,4,4,4,4,4,6],[6,6,6,6,6,6,6,6,6,6,4,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,4,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,4,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,4,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,4,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,4,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,4,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,4,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,4,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6],[6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6]],"output":[[4,4,4,4,4,6,4,4,4,4,4],[4,9,9,9,6,6,6,9,9,9,4],[4,9,0,0,6,6,6,0,0,9,4],[4,9,0,3,3,6,3,3,0,9,4],[4,6,6,3,8,8,8,3,6,6,4],[6,6,6,6,8,6,8,6,6,6,6],[4,6,6,3,8,8,8,3,6,6,4],[4,9,0,3,3,6,3,3,0,9,4],[4,9,0,0,6,6,6,0,0,9,4],[4,9,9,9,6,6,6,9,9,9,4],[4,4,4,4,4,6,4,4,4,4,4]]},{"input":[[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,5,2,2,1,1,1,2,8,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,1,2,1,2,8,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,8,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,8,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,4,4,2,2,2,4,4,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,0,0,0,2,2,2,0,0,0,2,2,2],[2,2,2,2,2,2,2,2,2,2,7,2,2,2,2,2,2,2,2,2,7,2],[2,2,2,2,2,2,2,2,2,2,7,2,2,2,2,2,2,2,2,2,7,2],[2,2,2,2,2,2,2,2,2,2,7,2,2,2,2,2,2,2,2,2,7,2],[2,2,2,2,2,2,2,2,2,2,7,7,7,7,2,2,2,7,7,7,7,2],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,0,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,0,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,0,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2]],"output":[[7,7,7,7,2,2,2,7,7,7,7],[7,0,0,0,2,2,2,0,0,0,7],[7,0,4,4,2,2,2,4,4,0,7],[7,0,4,8,8,2,8,8,4,0,7],[2,2,2,8,1,1,1,8,2,2,2],[2,2,2,2,1,5,1,2,2,2,2],[2,2,2,8,1,1,1,8,2,2,2],[7,0,4,8,8,2,8,8,4,0,7],[7,0,4,4,2,2,2,4,4,0,7],[7,0,0,0,2,2,2,0,0,0,7],[7,7,7,7,2,2,2,7,7,7,7]]},{"input":[[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,7,5,5,5,8,8,8,5,5,9,9,5,9,9,5,5,5,5,5,5,5],[5,5,5,5,5,8,5,8,5,5,9,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,8,8,8,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,9,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,9,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,6,6,6,5,6,6,6,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,0,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,0,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,0,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,0,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,0,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,0,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,0,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,0,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5]],"output":[[0,0,0,0,5,0,0,0,0],[0,6,6,6,5,6,6,6,0],[0,6,9,9,5,9,9,6,0],[0,6,9,8,8,8,9,6,0],[5,5,5,8,7,8,5,5,5],[0,6,9,8,8,8,9,6,0],[0,6,9,9,5,9,9,6,0],[0,6,6,6,5,6,6,6,0],[0,0,0,0,5,0,0,0,0]]},{"input":[[3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3],[3,2,2,2,3,3,4,4,3,4,4,3,7,3,3,3,3,3,3,3,3,3],[3,2,3,2,3,3,4,3,3,3,4,3,7,3,3,3,3,3,3,3,3,3],[3,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,4,3,3,3,4,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,4,4,3,4,4,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,3,3,7,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,3,3,7,3,3,3,3,3,3,3,3,3],[3,5,5,5,3,3,3,5,5,5,3,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,8,8,8,8,8,3,8,8,8,8,8,3],[3,3,3,3,3,3,3,3,3,3,8,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,8,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,8,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,8,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,8,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,8,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,8,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,8,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,8,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3]],"output":[[8,8,8,8,8,3,8,8,8,8,8],[8,5,5,5,3,3,3,5,5,5,8],[8,5,7,7,3,3,3,7,7,5,8],[8,5,7,4,4,3,4,4,7,5,8],[8,3,3,4,2,2,2,4,3,3,8],[3,3,3,3,2,3,2,3,3,3,3],[8,3,3,4,2,2,2,4,3,3,8],[8,5,7,4,4,3,4,4,7,5,8],[8,5,7,7,3,3,3,7,7,5,8],[8,5,5,5,3,3,3,5,5,5,8],[8,8,8,8,8,3,8,8,8,8,8]]},{"input":[[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,1,9,9,0,0,0,9,2,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,0,9,0,9,2,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,0,0,0,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,2,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,2,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,8,8,9,9,9,8,8,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,7,7,7,9,9,9,7,7,7,9,9,9],[9,9,9,9,9,9,9,9,9,9,6,9,9,9,9,9,9,9,9,9,6,9],[9,9,9,9,9,9,9,9,9,9,6,9,9,9,9,9,9,9,9,9,6,9],[9,9,9,9,9,9,9,9,9,9,6,9,9,9,9,9,9,9,9,9,6,9],[9,9,9,9,9,9,9,9,9,9,6,6,6,6,9,9,9,6,6,6,6,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,7,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,7,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,7,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9]],"output":[[6,6,6,6,9,9,9,6,6,6,6],[6,7,7,7,9,9,9,7,7,7,6],[6,7,8,8,9,9,9,8,8,7,6],[6,7,8,2,2,9,2,2,8,7,6],[9,9,9,2,0,0,0,2,9,9,9],[9,9,9,9,0,1,0,9,9,9,9],[9,9,9,2,0,0,0,2,9,9,9],[6,7,8,2,2,9,2,2,8,7,6],[6,7,8,8,9,9,9,8,8,7,6],[6,7,7,7,9,9,9,7,7,7,6],[6,6,6,6,9,9,9,6,6,6,6]]}]')
OUT_DIR = Path.cwd() / f"{TASK_ID}_compact_v4"
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_validation_summary_v4.json"
SUBMISSION_PATH = Path.cwd() / "submission.zip"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("task:", TASK_PATH)


task: /kaggle/input/competitions/neurogolf-2026/task096.json


In [6]:
def encode_grid(grid):
    arr = np.asarray(grid, dtype=np.int64)
    h, w = arr.shape
    assert h <= N and w <= N, (h, w)
    out = np.zeros((1, K, N, N), dtype=np.float32)
    one_hot = np.eye(K, dtype=np.float32)[arr]
    out[0, :, :h, :w] = one_hot.transpose(2, 0, 1)
    return out

def expected_tensor(grid):
    return encode_grid(grid)

def tensor_contract(y, grid):
    h, w = len(grid), len(grid[0])
    target = expected_tensor(grid)
    outside = y.copy()
    outside[:, :, :h, :w] = 0
    return {
        "exact": bool(np.array_equal(y, target)),
        "outside_zero": bool(np.all(outside == 0)),
        "inside_one_hot": bool(np.all(y[:, :, :h, :w].sum(axis=1) == 1.0)),
    }

def stable_hash(example):
    payload = json.dumps(example["input"], separators=(",", ":")).encode()
    return hashlib.sha256(payload).hexdigest()


In [7]:
arc_gen_all = task.get("arc-gen", [])
arc_gen_compatible = [e for e in arc_gen_all if max(len(e["input"]), len(e["input"][0]), len(e["output"]), len(e["output"][0])) <= N]
arc_gen_sorted = sorted(arc_gen_compatible, key=stable_hash)
holdout_n = int(np.ceil(0.60 * len(arc_gen_sorted)))
arc_gen_development = arc_gen_sorted[:-holdout_n] if holdout_n else arc_gen_sorted
arc_gen_holdout = arc_gen_sorted[-holdout_n:] if holdout_n else []

print("train/test/arc-gen:", len(task["train"]), len(task["test"]), len(arc_gen_all))
print("60% evaluation:", len(arc_gen_holdout), "adversarial:", len(ADVERSARIAL_CASES))


train/test/arc-gen: 3 1 262
60% evaluation: 158 adversarial: 6


In [8]:
class Base(nn.Module):
 def __init__(self):
  super().__init__()
  coord=torch.arange(N,dtype=torch.float32)
  self.register_buffer('coord',coord)
  self.register_buffer('colors',torch.arange(K,dtype=torch.float32))
  self.register_buffer('rr',coord.view(N,1).expand(N,N))
  self.register_buffer('cc',coord.view(1,N).expand(N,N))
 def canvas(self,h,w):return (self.rr[None]<h[:,None,None])&(self.cc[None]<w[:,None,None])

class Task096Compact(Base):
 def __init__(self,maxS=19):
  super().__init__();self.maxS=maxS
  self.sizes=list(range(1,maxS+1,2))
  # grouped conv kernels registered per candidate
  for S in self.sizes:
   ts=[1] if S==1 else list(range(2,(S+1)//2+1))
   for t in ts:
    p=torch.zeros((S,S),dtype=torch.float32)
    if S==1:p[0,0]=1
    else:
     for r in range(S):
      for c in range(S):
       if ((r in (0,S-1) and (c<t or c>=S-t)) or (c in (0,S-1) and (r<t or r>=S-t))):p[r,c]=1
    self.register_buffer(f'k_{S}_{t}',p.view(1,1,S,S).repeat(K,1,1,1))
 def forward(self,x):
  counts=x.sum((2,3))
  bg=counts.argmax(1).float()
  active_color=(counts>0.5)&(self.colors.view(1,K)!=bg[:,None])
  n=active_color.float().sum(1)
  has_single=((counts==1)&active_color).any(1)
  valid_sizes=[]; min_t=[]
  for S in self.sizes:
   ts=[1] if S==1 else list(range(2,(S+1)//2+1))
   vals=[]
   for t in ts:
    k=getattr(self,f'k_{S}_{t}')
    ov=F.conv2d(x,k,padding=S-1,groups=K)
    vmax=ov.amax((2,3))
    vals.append((vmax==counts)&active_color)
   vs=torch.stack(vals,2)
   valid_sizes.append(vs.any(2))
   # first valid t
   earlier=torch.cumsum(vs.float(),2)-vs.float()
   first=vs&(earlier==0)
   tv=torch.tensor(ts,dtype=torch.float32,device=x.device).view(1,1,-1)
   min_t.append((first.float()*tv).sum(2))
  valid=torch.stack(valid_sizes,2) # B,C,L
  t_by_size=torch.stack(min_t,2)
  L=len(self.sizes)
  svals=torch.tensor(self.sizes,dtype=torch.float32,device=x.device)
  # active target sizes: 1..2n-1 if singleton, else 3..2n+1
  rank=torch.arange(L,dtype=torch.float32,device=x.device)
  targ_single=rank[None,:]<n[:,None]
  targ_no=(rank[None,:]>=1)&(rank[None,:]<=n[:,None])
  target=(has_single[:,None]&targ_single)|((~has_single[:,None])&targ_no)
  cand=valid&target[:,None,:]&active_color[:,:,None]
  assigned=torch.zeros_like(cand)
  for _ in range(10):
   usedc=assigned.any(2);useds=assigned.any(1)
   work=cand&(~usedc[:,:,None])&(~useds[:,None,:])
   rd=work.float().sum(2);cd=work.float().sum(1)
   forced=work&((rd==1)[:,:,None]|(cd==1)[:,None,:])
   assigned=assigned|forced
  remc=active_color&(~assigned.any(2)); rems=target&(~assigned.any(1))
  # bbox max + unique columns tie-break among unresolved colors
  rowany=x.bool().any(3);colany=x.bool().any(2)
  big=torch.full_like(self.coord,1000.0);small=torch.full_like(self.coord,-1000.0)
  rmin=torch.where(rowany,self.coord.view(1,1,N),big.view(1,1,N)).amin(2)
  rmax=torch.where(rowany,self.coord.view(1,1,N),small.view(1,1,N)).amax(2)
  cmin=torch.where(colany,self.coord.view(1,1,N),big.view(1,1,N)).amin(2)
  cmax=torch.where(colany,self.coord.view(1,1,N),small.view(1,1,N)).amax(2)
  bmax=torch.maximum(rmax-rmin+1,cmax-cmin+1)
  uc=colany.float().sum(2)
  # rank = number remaining with lexicographically smaller (bmax,uc,color)
  ai=bmax[:,:,None];aj=bmax[:,None,:]
  ui=uc[:,:,None];uj=uc[:,None,:]
  ci=self.colors.view(1,K,1);cj=self.colors.view(1,1,K)
  jless=(aj<ai)|((aj==ai)&((uj<ui)|((uj==ui)&(cj<ci))))
  crank=(jless&remc[:,None,:]).float().sum(2)
  srank=torch.cumsum(rems.float(),1)-1.0
  tie=remc[:,:,None]&rems[:,None,:]&(crank[:,:,None]==srank[:,None,:])&cand
  assigned=assigned|tie
  # fallback: rank assignment without candidate only if needed
  stillc=active_color&(~assigned.any(2));stills=target&(~assigned.any(1))
  crank2=(jless&stillc[:,None,:]).float().sum(2)
  srank2=torch.cumsum(stills.float(),1)-1.0
  assigned=assigned|(stillc[:,:,None]&stills[:,None,:]&(crank2[:,:,None]==srank2[:,None,:]))
  Sval=(assigned.float()*svals.view(1,1,L)).sum(2)
  tval=(assigned.float()*t_by_size).sum(2)
  Smax=Sval.amax(1).clamp(min=1)
  shift=(Smax[:,None]-Sval)/2.0
  rr=self.rr.view(1,1,N,N);cc=self.cc.view(1,1,N,N)
  lr=rr-shift[:,:,None,None];lc=cc-shift[:,:,None,None]
  inside=(lr>=0)&(lc>=0)&(lr<Sval[:,:,None,None])&(lc<Sval[:,:,None,None])&active_color[:,:,None,None]
  topbot=(lr==0)|(lr==Sval[:,:,None,None]-1)
  leftright=(lc==0)|(lc==Sval[:,:,None,None]-1)
  armh=(lc<tval[:,:,None,None])|(lc>=Sval[:,:,None,None]-tval[:,:,None,None])
  armv=(lr<tval[:,:,None,None])|(lr>=Sval[:,:,None,None]-tval[:,:,None,None])
  masks=inside&((topbot&armh)|(leftright&armv))
  canv=self.canvas(Smax,Smax)
  occupied=masks.any(1)
  bgoh=(self.colors.view(1,K)==bg[:,None]).float()
  out=masks.float()+bgoh[:,:,None,None]*(canv&(~occupied))[:,None].float()
  return out*canv[:,None].float()


model = Task096Compact(maxS=11).eval()


In [9]:
def validate_eager(name, examples):
    bad = []
    with torch.no_grad():
        for i, ex in enumerate(examples):
            y = model(torch.from_numpy(encode_grid(ex["input"]))).cpu().numpy()
            if not np.array_equal(y, expected_tensor(ex["output"])):
                bad.append(i)
    print(name, len(examples)-len(bad), "/", len(examples), "bad", bad[:10])
    assert not bad, (name, bad[:10])

validate_eager("eager/train", task["train"])
validate_eager("eager/test", task["test"])
validate_eager("eager/adversarial", ADVERSARIAL_CASES)


eager/train 3 / 3 bad []
eager/test 1 / 1 bad []
eager/adversarial 6 / 6 bad []


In [10]:
dummy = torch.from_numpy(encode_grid(task["train"][0]["input"]))
with torch.no_grad():
    eager_dummy = model(dummy)
assert list(eager_dummy.shape) == [1, 10, 30, 30]
torch.onnx.export(
    model, dummy, str(ONNX_PATH),
    input_names=["input"], output_names=["output"],
    opset_version=18, do_constant_folding=True,
    dynamic_axes=None, dynamo=False, external_data=False,
)
onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
inferred = shape_inference.infer_shapes(onnx_model)
ops = Counter(node.op_type for node in onnx_model.graph.node)
FORBIDDEN = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
EXTENDED_AVOID = {"Einsum", "ScatterElements", "ScatterND", "GatherElements"}
forbidden = sorted(set(ops) & FORBIDDEN)
extended_present = sorted(set(ops) & EXTENDED_AVOID)
model_size = ONNX_PATH.stat().st_size
node_count = len(onnx_model.graph.node)
function_count = len(onnx_model.functions)
print("model bytes:", model_size, "nodes:", node_count)
print("operators:", dict(ops))
print("forbidden:", forbidden, "extended avoid:", extended_present)
assert model_size < 1_400_000
assert not forbidden
assert not extended_present
assert function_count == 0
assert node_count < 1000

del dummy, eager_dummy, model
gc.collect()
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass


/tmp/ipykernel_16/3810285047.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/tmp/ipykernel_16/389425973.py:46: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  tv=torch.tensor(ts,dtype=torch.float32,device=x.device).view(1,1,-1)
/tmp/ipykernel_16/389425973.py:51: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you 

model bytes: 131797 nodes: 819
operators: {'Constant': 256, 'ReduceSum': 67, 'ArgMax': 1, 'Cast': 67, 'Greater': 35, 'Unsqueeze': 100, 'Equal': 53, 'Not': 27, 'And': 81, 'Conv': 16, 'ReduceMax': 19, 'Concat': 8, 'CumSum': 8, 'Sub': 15, 'Mul': 10, 'Less': 9, 'LessOrEqual': 1, 'Or': 30, 'Where': 4, 'ReduceMin': 2, 'Add': 3, 'Max': 1, 'Clip': 1, 'Div': 1, 'GreaterOrEqual': 4}
forbidden: [] extended avoid: []


In [11]:
so = ort.SessionOptions()
so.intra_op_num_threads = 1
so.inter_op_num_threads = 1
so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
session = ort.InferenceSession(str(ONNX_PATH), sess_options=so, providers=["CPUExecutionProvider"])
input_shape = list(session.get_inputs()[0].shape)
output_shape = list(session.get_outputs()[0].shape)
assert input_shape == [1, 10, 30, 30], input_shape
assert output_shape == [1, 10, 30, 30], output_shape

def validate_onnx(name, examples):
    exact = outside = onehot = 0
    bad = []
    for i, ex in enumerate(examples):
        y = session.run(None, {"input": encode_grid(ex["input"])})[0]
        c = tensor_contract(y, ex["output"])
        exact += int(c["exact"]); outside += int(c["outside_zero"]); onehot += int(c["inside_one_hot"])
        if not c["exact"]:
            bad.append(i)
    result = {"ok": exact, "total": len(examples), "outside_zero_ok": outside, "inside_one_hot_ok": onehot, "bad_first10": bad[:10]}
    print(name, result)
    assert exact == len(examples), (name, bad[:10])
    assert outside == len(examples)
    assert onehot == len(examples)
    return result

# Runtime is part of this revision: the failed v3 graphs were much more expensive.
bench_x = encode_grid(task["test"][0]["input"])
for _ in range(10):
    session.run(None, {"input": bench_x})
t0 = time.perf_counter()
for _ in range(100):
    session.run(None, {"input": bench_x})
avg_ms = (time.perf_counter() - t0) * 10.0
print("average ONNX latency ms:", avg_ms)
assert avg_ms < 100.0

summary = {
    "task_id": TASK_ID,
    "revision": REVISION,
    "task_type": 'nonlocal nested broken-frame reconstruction',
    "structural_rule": 'Each non-background color is a fragment of a centered odd square frame whose four sides retain equal corner arms. Use grouped convolutions to test the parametric frame family, assign consecutive odd layer sizes by statically unrolled bipartite constraint propagation, select the minimum compatible arm length, and render the nested frames analytically.',
    "modelling_change": 'The v3 exhaustive translated C4 search was both over-general and extremely expensive. This model uses the actual frame family and global layer-order constraint.',
    "input_shape": input_shape,
    "output_shape": output_shape,
    "onnx_size_bytes": model_size,
    "onnx_node_count": node_count,
    "average_latency_ms": avg_ms,
    "ops": dict(ops),
    "forbidden_ops": forbidden,
    "extended_avoid_ops": extended_present,
    "function_count": function_count,
    "train": validate_onnx("onnx/train", task["train"]),
    "test": validate_onnx("onnx/test", task["test"]),
    "arc_gen_holdout_60pct": validate_onnx("onnx/arc-gen 60%", arc_gen_holdout),
    "arc_gen_all_diagnostic": validate_onnx("onnx/arc-gen all", arc_gen_compatible),
    "adversarial": validate_onnx("onnx/adversarial", ADVERSARIAL_CASES),
}
with SUMMARY_PATH.open("w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))


average ONNX latency ms: 11.225637649999953
onnx/train {'ok': 3, 'total': 3, 'outside_zero_ok': 3, 'inside_one_hot_ok': 3, 'bad_first10': []}
onnx/test {'ok': 1, 'total': 1, 'outside_zero_ok': 1, 'inside_one_hot_ok': 1, 'bad_first10': []}
onnx/arc-gen 60% {'ok': 158, 'total': 158, 'outside_zero_ok': 158, 'inside_one_hot_ok': 158, 'bad_first10': []}
onnx/arc-gen all {'ok': 262, 'total': 262, 'outside_zero_ok': 262, 'inside_one_hot_ok': 262, 'bad_first10': []}
onnx/adversarial {'ok': 6, 'total': 6, 'outside_zero_ok': 6, 'inside_one_hot_ok': 6, 'bad_first10': []}
{
  "task_id": "task096",
  "revision": "compact-v4-after-repeated-zero-score",
  "task_type": "nonlocal nested broken-frame reconstruction",
  "structural_rule": "Each non-background color is a fragment of a centered odd square frame whose four sides retain equal corner arms. Use grouped convolutions to test the parametric frame family, assign consecutive odd layer sizes by statically unrolled bipartite constraint propagation, s

In [12]:
if SUBMISSION_PATH.exists():
    SUBMISSION_PATH.unlink()
with zipfile.ZipFile(SUBMISSION_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")
with zipfile.ZipFile(SUBMISSION_PATH) as zf:
    assert zf.namelist() == [f"{TASK_ID}.onnx"]
    info = zf.infolist()[0]
    print("submission.zip:", info.filename, info.file_size, "bytes")


submission.zip: task096.onnx 131797 bytes


In [13]:
SUBMISSION_PATH

PosixPath('/kaggle/working/submission.zip')